In [ ]:
# pip install PyYAML

In [3]:
import os
import json
import shutil
from pathlib import Path
import yaml
import random
from tqdm import tqdm

# 1. 경로 설정
origin_root = Path("origin_sample_dataset") # <- 여기에 원본데이터셋 경로설정
yolo_root = Path("YOLOv11Dataset") # <- 여기에 Yolo v11 변환데이터셋 경로설정
yolo_root.mkdir(parents=True, exist_ok=True)

# 2. YOLO 디렉토리 구조 생성 (labels만 생성, images는 사용자가 별도로 관리)
dirs = [
    "labels/train",
    "labels/val",
    "labels/test"
]

for d in dirs:
    (yolo_root / d).mkdir(parents=True, exist_ok=True)

# 3. 클래스 매핑 정보 생성
class_mapping = {
    ("01", 0): 0, # '01' (배) + disease 0 = 0: '배 정상'
    ("01", 1): 1, # '01' (배) + disease 1 = 1: '배검은별무늬병'
    ("01", 2): 2, # '01' (배) + disease 2 = 2: '배과수화상병'

    ("02", 0): 8, # '02' (사과) + disease 0 = 8: '사과 정상'
    ("02", 3): 3, # '02' (사과) + disease 3 = 3: '사과갈색무늬병'
    ("02", 4): 4, # '02' (사과) + disease 4 = 4: '사과과수화상병'
    ("02", 5): 5, # '02' (사과) + disease 5 = 5: '사과부란병'
    ("02", 6): 6, # '02' (사과) + disease 6 = 6: '사과점무늬낙엽병'
    ("02", 7): 7  # '02' (사과) + disease 7 = 7: '사과탄저병'
}

# JSON → YOLO 레이블 변환 함수
def convert_label(json_path, img_filename_stem, img_width, img_height):
    """
    JSON 파일을 읽어 YOLO 형식의 텍스트 라벨을 반환합니다.
    Args:
        json_path (Path): JSON 파일의 전체 경로.
        img_filename_stem (str): 이미지 파일명 (확장자 제외), JSON 파일명과 동일하다고 가정.
        img_width (int): 원본 이미지의 너비.
        img_height (int): 원본 이미지의 높이.
    Returns:
        str: YOLO 형식의 라벨 문자열 (각 라인마다 하나의 객체), 또는 변환 실패 시 빈 문자열.
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 1. 파일명에서 작물 정보 추출
    # 예: V006_80_0_00_01_01_25_0_b06_20201005_0002_S01_1.jpg.json -> '01' (배)
    # _ 기준으로 5번째 위치 (인덱스 4)
    filename_parts = img_filename_stem.split('_')
    if len(filename_parts) > 4:
        crop_code = filename_parts[4] # '01' (배) 또는 '02' (사과)
    else:
        # print(f"    ! Warning: Could not extract crop code from filename: {img_filename_stem}. Skipping.")
        return "" # 유효하지 않은 파일명이면 빈 문자열 반환

    # 2. JSON에서 질병 정보 추출
    # 'annotations' -> 'disease' 필드 값 (정수형)
    json_disease_value = data['annotations']['disease']

    # 3. class_mapping을 사용하여 최종 class_id 결정
    class_key = (crop_code, json_disease_value)
    class_id = class_mapping.get(class_key) # 매핑에 없으면 None 반환
    
    if class_id is None:
        return "" # 매핑 정보가 없으면 빈 문자열 반환

    # 바운딩 박스 처리
    bbox_lines = []
    for point in data['annotations']['points']:
        xtl = point['xtl']
        ytl = point['ytl']
        xbr = point['xbr']
        ybr = point['ybr']
        
        # YOLO 형식으로 정규화 (x_center y_center width height)
        x_center = ((xtl + xbr) / 2) / img_width
        y_center = ((ytl + ybr) / 2) / img_height
        width = (xbr - xtl) / img_width
        height = (ybr - ytl) / img_height
        
        bbox_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
    
    return "\n".join(bbox_lines)

# 개별 JSON 파일 처리 함수
def process_json(json_file: Path, target_type: str):
    """
    단일 JSON 파일을 읽어 YOLO .txt 라벨 파일을 생성합니다.
    Args:
        json_file (Path): 처리할 JSON 파일의 Path 객체.
        target_type (str): 'train', 'val', 'test' 중 하나.
    """
    img_filename_stem = json_file.stem # JSON 파일명 (확장자 제외)

    # JSON에서 이미지 크기 추출 (description 필드 사용)
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
        
        img_width = json_data['description']['width']
        img_height = json_data['description']['height']
    except KeyError as e:
        # print(f"    ! Error: Missing key {e} in JSON file {json_file}. Skipping.")
        return
    except Exception as e:
        # print(f"    ! Error reading JSON for {json_file}: {e}. Skipping.")
        return

    # 레이블 변환
    yolo_label = convert_label(json_file, img_filename_stem, img_width, img_height)
    
    # 변환된 라벨이 없으면 (예: 매핑 실패) 파일 생성 건너뛰기
    if not yolo_label:
        return

    # YOLO 형식 라벨 저장
    label_dir = yolo_root / "labels" / target_type
    txt_path = label_dir / f"{img_filename_stem}.txt" # JSON과 동일한 파일명으로 .txt 생성
    
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(yolo_label)

# 데이터셋 처리 함수 (tqdm 적용)
def process_dataset(src_type: str):
    """
    원본 데이터셋의 Training/Validation 폴더를 탐색하여 JSON 파일을 처리하고
    YOLO 형식의 .txt 라벨 파일을 생성합니다.
    Args:
        src_type (str): 'Training' 또는 'Validation'.
    """
    print(f"\nProcessing {src_type} data...")
    
    for folder in (origin_root / src_type).iterdir():
        # '[라벨]'로 시작하는 폴더만 처리
        if folder.name.startswith("[라벨]"):
            json_files = list(folder.glob("*.json"))
            random.shuffle(json_files) # 파일 목록 섞기

            if src_type == "Training":
                target_type = "train"
                print(f"  Converting JSONs from '{folder.name}' to '{target_type}' labels...")
                for json_file in tqdm(json_files, desc=f"  {target_type.upper()} labels"):
                    process_json(json_file, target_type)
            elif src_type == "Validation":
                # Validation 데이터는 val과 test로 5:5 분할
                split_idx = len(json_files) // 2
                val_files = json_files[:split_idx]
                test_files = json_files[split_idx:]
                
                print(f"  Converting JSONs from '{folder.name}' to 'val' labels...")
                for json_file in tqdm(val_files, desc="  VAL labels"):
                    process_json(json_file, "val")
                
                print(f"  Converting JSONs from '{folder.name}' to 'test' labels...")
                for json_file in tqdm(test_files, desc="  TEST labels"):
                    process_json(json_file, "test")

# 6. 데이터 처리 실행
process_dataset("Training")
process_dataset("Validation")

# 7. dataset.yaml 파일 생성
yaml_content = {
    'path': str(yolo_root.resolve()),
    # 이미지 경로는 사용자가 실제 이미지 위치에 맞춰 직접 설정해야 합니다.
    # 이 스크립트는 라벨만 생성하므로, 아래 경로는 placeholder입니다.
    'train': '../images/train', # 상대 경로 예시
    'val': '../images/val',     # YOLO 프로젝트 루트에 이미지 폴더가 있다면 이렇게 설정
    'test': '../images/test',   # 혹은 실제 절대/상대 경로 지정

    'names': { # 최종 9개 클래스 정의
        0: '배 정상',
        1: '배검은별무늬병',
        2: '배과수화상병',
        3: '사과갈색무늬병',
        4: '사과과수화상병',
        5: '사과부란병',
        6: '사과점무늬낙엽병',
        7: '사과탄저병',
        8: '사과 정상'
    }
}

with open(yolo_root / "dataset.yaml", 'w', encoding='utf-8') as f:
    yaml.dump(yaml_content, f, allow_unicode=True, sort_keys=False)

# 8. 분할 결과 통계 출력
def print_stats():
    print("\n---")
    print("Dataset Split Statistics (Labels Only):")
    for split in ['train', 'val', 'test']:
        label_count = len(list((yolo_root / "labels" / split).glob("*.txt")))
        print(f"  - {split}: {label_count} labels")
    print("---\n")

print("\nJSON to YOLO label conversion completed successfully!")
print(f"YOLO label files are generated in: {yolo_root / 'labels'}")
print(f"The 'dataset.yaml' file is located at: {yolo_root / 'dataset.yaml'}")
print_stats()

print("Important: Remember to correctly configure 'train', 'val', and 'test' paths in 'dataset.yaml'")
print("to point to your actual image directories for YOLOv11 training.")


Processing Training data...
  Converting JSONs from '[라벨]배_0.정상' to 'train' labels...


  TRAIN labels: 100%|██████████| 20434/20434 [03:52<00:00, 88.05it/s] 


  Converting JSONs from '[라벨]배_1.질병' to 'train' labels...


  TRAIN labels: 100%|██████████| 2557/2557 [00:28<00:00, 90.52it/s] 


  Converting JSONs from '[라벨]사과_0.정상' to 'train' labels...


...

  Converting JSONs from '[라벨]사과_1.질병' to 'val' labels...


  VAL labels: 100%|██████████| 518/518 [00:05<00:00, 102.48it/s]


  Converting JSONs from '[라벨]사과_1.질병' to 'test' labels...


  TEST labels: 100%|██████████| 518/518 [00:04<00:00, 110.16it/s]



JSON to YOLO label conversion completed successfully!
YOLO label files are generated in: YOLOv11Dataset\labels
The 'dataset.yaml' file is located at: YOLOv11Dataset\dataset.yaml

---
Dataset Split Statistics (Labels Only):
  - train: 56790 labels
  - val: 3542 labels
  - test: 3562 labels
---

Important: Remember to correctly configure 'train', 'val', and 'test' paths in 'dataset.yaml'
to point to your actual image directories for YOLOv11 training.


In [4]:
import os
import json
import shutil
from pathlib import Path
import yaml
import random
from tqdm import tqdm

# 1. 경로 설정
origin_root = Path("origin_sample_dataset") # <- 여기에 원본데이터셋 경로설정
yolo_root = Path("YOLOv11Dataset") # <- 여기에 Yolo v11 변환데이터셋 경로설정
yolo_root.mkdir(parents=True, exist_ok=True)

# 2. YOLO 디렉토리 구조 생성 (labels만 생성, images는 사용자가 별도로 관리)
dirs = [
    "labels/train",
    "labels/val",
    "labels/test"
]

for d in dirs:
    (yolo_root / d).mkdir(parents=True, exist_ok=True)

# 3. 클래스 매핑 정보 생성
class_mapping = {
    ("01", 0): 0, # '01' (배) + disease 0 = 0: '배 정상'
    ("01", 1): 1, # '01' (배) + disease 1 = 1: '배검은별무늬병'
    ("01", 2): 2, # '01' (배) + disease 2 = 2: '배과수화상병'

    ("02", 0): 8, # '02' (사과) + disease 0 = 8: '사과 정상'
    ("02", 1): 3, # '02' (사과) + disease 1 = 3: '사과갈색무늬병'
    ("02", 2): 4, # '02' (사과) + disease 2 = 4: '사과과수화상병'
    ("02", 3): 5, # '02' (사과) + disease 3 = 5: '사과부란병'
    ("02", 4): 6, # '02' (사과) + disease 4 = 6: '사과점무늬낙엽병'
    ("02", 5): 7  # '02' (사과) + disease 5 = 7: '사과탄저병'
}

# JSON → YOLO 레이블 변환 함수
def convert_label(json_path, img_filename_stem, img_width, img_height):
    """
    JSON 파일을 읽어 YOLO 형식의 텍스트 라벨을 반환합니다.
    Args:
        json_path (Path): JSON 파일의 전체 경로.
        img_filename_stem (str): 이미지 파일명 (확장자 제외), JSON 파일명과 동일하다고 가정.
        img_width (int): 원본 이미지의 너비.
        img_height (int): 원본 이미지의 높이.
    Returns:
        str: YOLO 형식의 라벨 문자열 (각 라인마다 하나의 객체), 또는 변환 실패 시 빈 문자열.
    """
    with open(json_path, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    # 1. 파일명에서 작물 정보 추출
    # 예: V006_80_0_00_01_01_25_0_b06_20201005_0002_S01_1.jpg.json -> '01' (배)
    # _ 기준으로 5번째 위치 (인덱스 4)
    filename_parts = img_filename_stem.split('_')
    if len(filename_parts) > 4:
        crop_code = filename_parts[4] # '01' (배) 또는 '02' (사과)
    else:
        print(f"    ! Warning: Could not extract crop code from filename: {img_filename_stem} (JSON: {json_path.name}). Skipping due to invalid filename format.")
        return "" # 유효하지 않은 파일명이면 빈 문자열 반환

    # 2. JSON에서 질병 정보 추출
    # 'annotations' -> 'disease' 필드 값 (정수형)
    json_disease_value = data['annotations']['disease']

    # 3. class_mapping을 사용하여 최종 class_id 결정
    class_key = (crop_code, json_disease_value)
    class_id = class_mapping.get(class_key) # 매핑에 없으면 None 반환
    
    if class_id is None:
        print(f"    ! Warning: No class_id mapping found for crop_code '{crop_code}' and JSON disease value '{json_disease_value}' in {img_filename_stem} (JSON: {json_path.name}). Skipping.")
        return "" # 매핑 정보가 없으면 빈 문자열 반환

    # 바운딩 박스 처리
    bbox_lines = []
    # 'annotations' 내에 'points' 키가 없을 경우도 처리 (에러 방지)
    if 'points' not in data['annotations'] or not data['annotations']['points']:
        print(f"    ! Warning: No bounding box 'points' found in JSON file: {json_path.name}. Skipping this label.")
        return "" # 바운딩 박스 정보가 없으면 빈 문자열 반환

    for point in data['annotations']['points']:
        xtl = point['xtl']
        ytl = point['ytl']
        xbr = point['xbr']
        ybr = point['ybr']
        
        # YOLO 형식으로 정규화 (x_center y_center width height)
        x_center = ((xtl + xbr) / 2) / img_width
        y_center = ((ytl + ybr) / 2) / img_height
        width = (xbr - xtl) / img_width
        height = (ybr - ytl) / img_height
        
        bbox_lines.append(f"{class_id} {x_center:.6f} {y_center:.6f} {width:.6f} {height:.6f}")
    
    return "\n".join(bbox_lines)

# 개별 JSON 파일 처리 함수
def process_json(json_file: Path, target_type: str):
    """
    단일 JSON 파일을 읽어 YOLO .txt 라벨 파일을 생성합니다.
    Args:
        json_file (Path): 처리할 JSON 파일의 Path 객체.
        target_type (str): 'train', 'val', 'test' 중 하나.
    """
    img_filename_stem = json_file.stem # JSON 파일명 (확장자 제외)

    # JSON에서 이미지 크기 추출 (description 필드 사용)
    try:
        with open(json_file, 'r', encoding='utf-8') as f:
            json_data = json.load(f)
        
        img_width = json_data['description']['width']
        img_height = json_data['description']['height']
    except KeyError as e:
        print(f"    ! Error: Missing key '{e}' in JSON file: {json_file.name} (Likely 'description' or 'width'/'height' within it). Skipping.")
        return
    except Exception as e:
        print(f"    ! Error reading JSON file {json_file.name}: {e}. Skipping.")
        return

    # 레이블 변환
    yolo_label = convert_label(json_file, img_filename_stem, img_width, img_height)
    
    # 변환된 라벨이 없으면 (예: 매핑 실패, 바운딩 박스 없음 등) 파일 생성 건너뛰기
    if not yolo_label:
        return

    # YOLO 형식 라벨 저장
    label_dir = yolo_root / "labels" / target_type
    txt_path = label_dir / f"{img_filename_stem}.txt" # JSON과 동일한 파일명으로 .txt 생성
    
    try:
        with open(txt_path, 'w', encoding='utf-8') as f:
            f.write(yolo_label)
    except Exception as e:
        print(f"    ! Error writing TXT file for {json_file.name}: {e}. Skipping.")

# 데이터셋 처리 함수 (tqdm 적용)
def process_dataset(src_type: str):
    """
    원본 데이터셋의 Training/Validation 폴더를 탐색하여 JSON 파일을 처리하고
    YOLO 형식의 .txt 라벨 파일을 생성합니다.
    Args:
        src_type (str): 'Training' 또는 'Validation'.
    """
    print(f"\nProcessing {src_type} data...")
    
    for folder in (origin_root / src_type).iterdir():
        # '[라벨]'로 시작하는 폴더만 처리
        if folder.name.startswith("[라벨]"):
            json_files = list(folder.glob("*.json"))
            random.shuffle(json_files) # 파일 목록 섞기

            if src_type == "Training":
                target_type = "train"
                print(f"  Converting JSONs from '{folder.name}' to '{target_type}' labels...")
                # tqdm에 total 인자를 명시적으로 전달하여 정확한 진행률 표시
                for json_file in tqdm(json_files, desc=f"  {target_type.upper()} labels", total=len(json_files)):
                    process_json(json_file, target_type)
            elif src_type == "Validation":
                # Validation 데이터는 val과 test로 5:5 분할
                split_idx = len(json_files) // 2
                val_files = json_files[:split_idx]
                test_files = json_files[split_idx:]
                
                print(f"  Converting JSONs from '{folder.name}' to 'val' labels...")
                for json_file in tqdm(val_files, desc="  VAL labels", total=len(val_files)):
                    process_json(json_file, "val")
                
                print(f"  Converting JSONs from '{folder.name}' to 'test' labels...")
                for json_file in tqdm(test_files, desc="  TEST labels", total=len(test_files)):
                    process_json(json_file, "test")

# 6. 데이터 처리 실행
process_dataset("Training")
process_dataset("Validation")

# 7. dataset.yaml 파일 생성
yaml_content = {
    'path': str(yolo_root.resolve()),
    # 이미지 경로는 사용자가 실제 이미지 위치에 맞춰 직접 설정해야 합니다.
    # 이 스크립트는 라벨만 생성하므로, 아래 경로는 placeholder입니다.
    'train': '../images/train', # 상대 경로 예시
    'val': '../images/val',     # YOLO 프로젝트 루트에 이미지 폴더가 있다면 이렇게 설정
    'test': '../images/test',   # 혹은 실제 절대/상대 경로 지정

    'names': { # 최종 9개 클래스 정의
        0: '배 정상',
        1: '배검은별무늬병',
        2: '배과수화상병',
        3: '사과갈색무늬병',
        4: '사과과수화상병',
        5: '사과부란병',
        6: '사과점무늬낙엽병',
        7: '사과탄저병',
        8: '사과 정상'
    }
}

with open(yolo_root / "dataset.yaml", 'w', encoding='utf-8') as f:
    yaml.dump(yaml_content, f, allow_unicode=True, sort_keys=False)

# 8. 분할 결과 통계 출력
def print_stats():
    print("\n---")
    print("Dataset Split Statistics (Labels Only):")
    for split in ['train', 'val', 'test']:
        label_count = len(list((yolo_root / "labels" / split).glob("*.txt")))
        print(f"  - {split}: {label_count} labels")
    print("---\n")

print("\nJSON to YOLO label conversion completed successfully!")
print(f"YOLO label files are generated in: {yolo_root / 'labels'}")
print(f"The 'dataset.yaml' file is located at: {yolo_root / 'dataset.yaml'}")
print_stats()

print("Important: Remember to correctly configure 'train', 'val', and 'test' paths in 'dataset.yaml'")
print("to point to your actual image directories for YOLOv11 training.")


Processing Training data...
  Converting JSONs from '[라벨]배_0.정상' to 'train' labels...


  TRAIN labels: 100%|██████████| 20434/20434 [02:00<00:00, 169.47it/s]


  Converting JSONs from '[라벨]배_1.질병' to 'train' labels...


  TRAIN labels:   6%|▌         | 158/2557 [00:00<00:01, 1568.60it/s]

    ! Warning: No class_id mapping found for crop_code '01' and JSON disease value '4' in V006_80_1_02_01_03_23_3_4142r_20200917_2.jpg (JSON: V006_80_1_02_01_03_23_3_4142r_20200917_2.jpg.json). Skipping.
    ! Warning: No class_id mapping found for crop_code '01' and JSON disease value '4' in V006_80_1_02_01_03_23_3_9177r_20201015_1.jpg (JSON: V006_80_1_02_01_03_23_3_9177r_20201015_1.jpg.json). Skipping.
    ! Warning: No class_id mapping found for crop_code '01' and JSON disease value '4' in V006_80_1_02_01_04_23_3_2181r_20200916_1.jpg (JSON: V006_80_1_02_01_04_23_3_2181r_20200916_1.jpg.json). Skipping.


...

  TEST labels:  78%|███████▊  | 405/518 [00:00<00:00, 1941.60it/s]

    ! Warning: No class_id mapping found for crop_code '02' and JSON disease value '6' in V006_80_1_06_02_03_23_2_5009y_20201005_7.jpg (JSON: V006_80_1_06_02_03_23_2_5009y_20201005_7.jpg.json). Skipping.
    ! Warning: No class_id mapping found for crop_code '02' and JSON disease value '7' in V006_80_1_07_02_01_23_2_5722b_20201102_3.jpg (JSON: V006_80_1_07_02_01_23_2_5722b_20201102_3.jpg.json). Skipping.
    ! Warning: No class_id mapping found for crop_code '02' and JSON disease value '7' in V006_80_1_07_02_01_23_2_4498w_20200916_70.JPG (JSON: V006_80_1_07_02_01_23_2_4498w_20200916_70.JPG.json). Skipping.
    ! Warning: No class_id mapping found for crop_code '02' and JSON disease value '6' in V006_80_1_06_02_03_22_2_4147y_20201028_228.jpg (JSON: V006_80_1_06_02_03_22_2_4147y_20201028_228.jpg.json). Skipping.
    ! Warning: No class_id mapping found for crop_code '02' and JSON disease value '7' in V006_80_1_07_02_01_23_2_5722b_20201012_37.jpg (JSON: V006_80_1_07_02_01_23_2_5722b_20201

  TEST labels: 100%|██████████| 518/518 [00:00<00:00, 1828.80it/s]

    ! Warning: No class_id mapping found for crop_code '02' and JSON disease value '6' in V006_80_1_06_02_03_23_2_4147y_20201008_76.jpg (JSON: V006_80_1_06_02_03_23_2_4147y_20201008_76.jpg.json). Skipping.
    ! Warning: No class_id mapping found for crop_code '02' and JSON disease value '6' in V006_80_1_06_02_03_23_2_7762y_20201020_170.jpg (JSON: V006_80_1_06_02_03_23_2_7762y_20201020_170.jpg.json). Skipping.
    ! Warning: No class_id mapping found for crop_code '02' and JSON disease value '7' in V006_80_1_07_02_01_23_2_5722b_20201121_5.jpg (JSON: V006_80_1_07_02_01_23_2_5722b_20201121_5.jpg.json). Skipping.
    ! Warning: No class_id mapping found for crop_code '02' and JSON disease value '7' in V006_80_1_07_02_01_23_2_5722b_20201010_24.jpg (JSON: V006_80_1_07_02_01_23_2_5722b_20201010_24.jpg.json). Skipping.
    ! Warning: No class_id mapping found for crop_code '02' and JSON disease value '7' in V006_80_1_07_02_01_23_2_1655w_20200929_18.JPG (JSON: V006_80_1_07_02_01_23_2_1655w_202

  - train: 56790 labels
  - val: 3550 labels
  - test: 3554 labels
---

Important: Remember to correctly configure 'train', 'val', and 'test' paths in 'dataset.yaml'
to point to your actual image directories for YOLOv11 training.
